In [ ]:
#ChatGPT
In studies of steel surface defect detection, preprocessing (contrast enhancement, grayscale conversion, filtering) has been shown to boost classification accuracy. For example: preprocessing grayscale images + using edge/texture operators (e.g., Sobel, Laplace) fused with original grayscale improved accuracy to ~99.77% on a steel‐plate defect dataset. 
AIMS Press

Also, one paper found that “image preprocessing with well-known filters can effectively increase the classification accuracy, above the values achievable by applying even the most fine-tuned CNN architectures only” on steel surface defects. 
SpringerLink

Converting to grayscale simplifies the input (removes color channel variability) and can reduce data size / simplify the model. In one steel-ball defect study they found a grayscale-based model had superior generalization and smaller size. 
Tech Science

Specific image processing techniques useful in this domain include: histogram equalization / CLAHE (contrast limited adaptive histogram equalization), denoising, morphological operations, edge/texture extraction (Sobel, Laplacian, LBP). 
MDPI
+2
MDPI
+2

Because defects may be small, subtle and appear in low contrast to the background, good preprocessing helps make them “pop out” for the classifier.

In [ ]:
#Chatgpt suggestions
Convert images to grayscale (or single‐channel) to simplify input.

Enhance contrast (CLAHE), denoise if needed, extract edges/textures (Sobel, Laplacian, LBP).

Stack processed channels as input features (or just use the best single channel).

Use data augmentation to increase robustness.

Use a suitable CNN classifier (or segmentation + classifier) and train with your preprocessed dataset.

Monitor improvements in accuracy/metrics as you try different preprocessing pipelines.

In [ ]:
ROI / Masking: If defects appear only on certain parts of steel panels (e.g., surface region, not edges), you might mask out irrelevant regions so the model focuses on relevant area.

Grayscale conversion is good: As shown by the GitHub implementation for the Severstal Steel Defect Detection dataset, they converted to grayscale (or 1‐channel) and found no loss in performance compared to 3‐channel input. 
GitHub

Edge and texture extraction help: Using Sobel/Laplacian or Local Binary Patterns (LBP) in preprocessing helps capture defect shapes/textures.

Contrast enhancement: Many steel surface images have low contrast between defect and background. Applying CLAHE or other histogram‐enhancement methods helps. 
MDPI
+1

Augmentation matters: Especially when defect examples are few / imbalanced. Use flips, rotations, shifts, brightness variation. In one steel defect classification paper they used horizontal + vertical flipping with probability 0.5. 
MDPI

Imbalanced classes: Defects may be rare. Strategies: oversample defect class, use balanced batches, use focal loss or other class‐imbalance aware losses.

Small defects detection: For very small defects, increasing input resolution, multi‐scale feature extraction, and careful preprocessing are important. 
MDPI
+1

Test preprocessing effects: Compare performance with/without certain preprocessing steps (grayscale only vs grayscale+CLAHE vs grayscale+edges) to see what helps your specific dataset.

Single‐channel input benefits: Less memory, simpler model, may speed up training and inference.

In [ ]:
# Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Helper: show image
def show_img(img, title=""):
    plt.figure(figsize=(4,4))
    plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

# 1. Load and convert to grayscale
def load_and_preprocess(path, target_size=(256,256)):
    # read color, convert to grayscale
    img = cv2.imread(path)  # BGR
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # resize
    img_gray = cv2.resize(img_gray, target_size)
    return img_gray

# Example
sample_path = 'path/to/your/image.jpg'
img_gray = load_and_preprocess(sample_path)
show_img(img_gray, title="Grayscale")

# 2. Contrast enhancement: CLAHE (good for industrial / low‐contrast images)
def apply_clahe(img_gray, clipLimit=2.0, tileGridSize=(8,8)):
    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    img_cl = clahe.apply(img_gray)
    return img_cl

img_cl = apply_clahe(img_gray)
show_img(img_cl, title="After CLAHE")

# 3. Denoise (optional, if images are noisy)
img_dn = cv2.fastNlMeansDenoising(img_cl, h=10, templateWindowSize=7, searchWindowSize=21)
show_img(img_dn, title="After Denoising")

# 4. Edge / texture enhancement (optional)
# e.g., using Sobel or Laplacian
sobelx = cv2.Sobel(img_dn, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img_dn, cv2.CV_64F, 0, 1, ksize=3)
sobel = cv2.magnitude(sobelx, sobely)
# normalize to 0-255
sobel = np.uint8(255 * (sobel / np.max(sobel)))
show_img(sobel, title="Sobel edges")

# 5. Combine channels (if you want multi‐channel input)
# Sometimes you might stack: [grayscale, CLAHE version, edges] as 3-channel
img_stack = np.stack([img_gray, img_cl, sobel], axis=-1)  # shape: H×W×3
# But if you want single‐channel, you can just keep img_cl or img_dn

# 6. Build dataset arrays
# Suppose you have folder structure: dataset/{defect, no_defect}/ images
data = []
labels = []
target_size = (256,256)

for label_name, label_idx in [('no_defect',0), ('defect',1)]:
    folder = os.path.join('dataset', label_name)
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        img_gray = load_and_preprocess(path, target_size=target_size)
        img_cl = apply_clahe(img_gray)
        img_dn = cv2.fastNlMeansDenoising(img_cl, h=10, templateWindowSize=7, searchWindowSize=21)
        # choose either single‐channel version
        # Or stack channels if model expects 3 channels
        img_in = img_dn  # single channel
        # optionally resize to include channel dimension
        img_in = img_in[..., np.newaxis]  # shape H×W×1
        data.append(img_in)
        labels.append(label_idx)

data = np.array(data, dtype='float32') / 255.0
labels = np.array(labels)

# 7. Train/test split
X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, stratify=labels, random_state=42)

# 8. Data augmentation (on grayscale)
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    # vertical_flip maybe if your domain allows
    zoom_range=0.1,
    brightness_range=(0.8,1.2)
)
# When using a grayscale single channel, you may need a custom generator or stack to 3 channels if your model expects 3.

# 9. Model definition (simple CNN)
input_shape = (target_size[0], target_size[1], 1)  # single channel
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# 10. Training
batch_size = 32
epochs = 20

# If using datagen, you can wrap it
train_gen = datagen.flow(X_train, y_train, batch_size=batch_size)
steps_per_epoch = len(X_train) // batch_size

history = model.fit(train_gen,
                    steps_per_epoch=steps_per_epoch,
                    epochs=epochs,
                    validation_data=(X_val, y_val))

# 11. Evaluate
val_loss, val_acc = model.evaluate(X_val, y_val)
print("Validation accuracy:", val_acc)

# 12. Visualization of results
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
